In [52]:
%load_ext autoreload
%autoreload 2

In [53]:
import json
import os, sys
import numpy as np
import pandas as pd

current_path = os.getcwd()
sys.path.append(os.path.join(current_path, '..'))

from adapt_model import parse_experiment_name
from utils.globals import *


In [54]:
model_name = 'llama2_13B'

num_layers_map = {
    'llama_7B': [1,3,5,7,8,9,10, 11],
    'llama2_13B': [3, 5, 7, 9, 11, 12, 14],
    'almar_13B': [3, 5, 7, 9, 11, 12, 14],
}

variable_parameters = {
    'num_layers': num_layers_map[model_name],
    'factual_thr': [0.01, 0.05, 0.1, 0.25, 1.0]
}



In [55]:
def load_data_file(model_name, method, experiment_name, test_file):
    if experiment_name and 'method' != 'original':
        result_dir = experiment_name
    else:
        result_dir = model_name
    result_file = os.path.join(RESULTS_DIR, method, result_dir, test_file)
    try:
        with open(result_file, 'r') as f:
            results = json.load(f)
    except FileNotFoundError:
        print(f"File {result_file} not found")
        results = None

    return results

Process generation results

In [56]:
def parse_generation_results(layers, fact_thr, results):
    out_row = {'layers': layers, 'threshold': fact_thr}
    for key in (
            #'slope_s', 'intercept_s', 'slope_f',
            'intercept_f', 'joint_slope_s', 'joint_slope_f', 'joint_intercept', 'prob_he', 'prob_she', 'prob_they', 'spearman_s', 'spearman_f'):
        if results is None:
            out_row[key] = 0.
        else:
            out_row[key] = results[key]
    return out_row

In [57]:
gen_res = pd.DataFrame(columns=['layers', 'threshold',
                                #'slope_s', 'intercept_s', 'slope_f', 'intercept_f',
                                'joint_slope_s', 'joint_slope_f', 'joint_intercept','prob_he', 'prob_she', 'prob_they',
                                'spearman_s', 'spearman_f'])
test_file = 'res_gen_test_dama.json'
# row = {'layers': None,'dimensions': None, 'slope_s': 0. ,'intercept_s': 0., 'slope_f': 0., 'intercept_f': 0.,
#        'prob_he': 0., 'prob_she': 0., 'prob_they': 0.}

tmp_res = load_data_file(model_name, 'original', None, test_file)

gen_res.loc[len(gen_res)] = parse_generation_results(0, 0.0, tmp_res)

for num_layers in variable_parameters['num_layers']:
    for fact_thr in variable_parameters['factual_thr']:
        if fact_thr == 1.00:
            experiment_name = f"{model_name}_{num_layers}L_factual"
        elif fact_thr == 0.0:
            experiment_name = f"{model_name}_{num_layers}L"
        else:
            experiment_name = f"{model_name}_{num_layers}L_factual_{fact_thr}"
        tmp_res = load_data_file(model_name, 'DAMA_L', experiment_name, test_file)
        if tmp_res is None:
            continue
        gen_res.loc[len(gen_res)] = parse_generation_results(num_layers, fact_thr, tmp_res)

In [58]:
display(gen_res)
# save to csv
gen_res.to_csv(os.path.join(RESULTS_DIR, 'gen_res_2dama.csv'), index=False)

Process causal lm results

In [59]:
def parse_causal_lm_results(layers, fact_thr, results):
    out_row = {'layers': layers, 'threshold':fact_thr, 'perplexity': results['mean_perplexity'] if results else 0.}
    return out_row

In [60]:
lm_res = pd.DataFrame(columns=['layers', 'threshold', 'perplexity'])

test_file = 'res_causal_lm_wikitext_wikitext-103-raw-v1.json'
# row = {'layers': None,'dimensions': None, 'perplexity': 0.}

tmp_res = load_data_file(model_name, 'original', None, test_file)
lm_res.loc[len(lm_res)] = parse_causal_lm_results(0, 0.0, tmp_res)

for num_layers in variable_parameters['num_layers']:
    for fact_thr in variable_parameters['factual_thr']:
        if fact_thr == 1.00:
            experiment_name = f"{model_name}_{num_layers}L_factual"
        elif fact_thr == 0.0:
            experiment_name = f"{model_name}_{num_layers}L"
        else:
            experiment_name = f"{model_name}_{num_layers}L_factual_{fact_thr}"
        tmp_res = load_data_file(model_name, 'DAMA_L', experiment_name, test_file)
        if tmp_res is None:
            continue
        lm_res.loc[len(lm_res)] = parse_causal_lm_results(num_layers, fact_thr, tmp_res)

In [61]:
display(lm_res)
# save to csv
lm_res.to_csv(os.path.join(RESULTS_DIR, 'lm_res_2dama.csv'), index=False)

Process coreference results

In [62]:
def parse_coref_results(layers, fact_thr, results_a1, results_a2, results_p1, results_p2):
    out_row = {'layers': layers, 'threshold': fact_thr}

    if results_a1 is None or results_a2 is None or results_p1 is None or results_p2 is None:
        out_row['anti_male'] = 0.
        out_row['anti_female'] = 0.
        out_row['pro_male'] = 0.
        out_row['pro_female'] = 0.
    else:
        out_row['anti_male'] = (results_a1['m_acc'] + results_a2['m_acc']) / 2.
        out_row['anti_female'] = (results_a1['f_acc'] + results_a2['f_acc']) / 2.
        out_row['pro_male'] = (results_p1['m_acc'] + results_p2['m_acc']) / 2.
        out_row['pro_female'] = (results_p1['f_acc'] + results_p2['f_acc']) / 2.

    out_row['ACC'] = (out_row['anti_male'] + out_row['anti_female'] + out_row['pro_male'] + out_row['pro_female']) / 4.
    out_row['Delta_S'] = (out_row['pro_male'] + out_row['pro_female'] - out_row['anti_male'] - out_row['anti_female']) / 2.
    out_row['Delta_G'] = (out_row['pro_male'] + out_row['anti_male'] - out_row['pro_female'] - out_row['anti_female']) / 2.
    return out_row

In [63]:
coref_res = pd.DataFrame(columns=['layers', 'threshold', 'ACC', 'Delta_S', 'Delta_G', 'anti_male', 'anti_female', 'pro_male', 'pro_female'])

test_file_a1 = 'res_coref_anti_type1_test.json'
test_file_a2 = 'res_coref_anti_type2_test.json'
test_file_p1 = 'res_coref_pro_type1_test.json'
test_file_p2 = 'res_coref_pro_type2_test.json'


tmp_res_a1 = load_data_file(model_name, 'original', None, test_file_a1)
tmp_res_a2 = load_data_file(model_name, 'original', None, test_file_a2)
tmp_res_p1 = load_data_file(model_name, 'original', None, test_file_p1)
tmp_res_p2 = load_data_file(model_name, 'original', None, test_file_p2)

coref_res.loc[len(coref_res)] = parse_coref_results(0, 0.0,tmp_res_a1, tmp_res_a2, tmp_res_p1, tmp_res_p2)

for num_layers in variable_parameters['num_layers']:
    for fact_thr in variable_parameters['factual_thr']:
        if fact_thr == 1.00:
            experiment_name = f"{model_name}_{num_layers}L_factual"
        elif fact_thr == 0.0:
            experiment_name = f"{model_name}_{num_layers}L"
        else:
            experiment_name = f"{model_name}_{num_layers}L_factual_{fact_thr}"
        
        tmp_res_a1 = load_data_file(model_name, 'DAMA_L', experiment_name, test_file_a1)
        tmp_res_a2 = load_data_file(model_name, 'DAMA_L', experiment_name, test_file_a2)
        tmp_res_p1 = load_data_file(model_name, 'DAMA_L', experiment_name, test_file_p1)
        tmp_res_p2 = load_data_file(model_name, 'DAMA_L', experiment_name, test_file_p2)
        if tmp_res_a1 is None:
            continue
        coref_res.loc[len(coref_res)] = parse_coref_results(num_layers, fact_thr, tmp_res_a1, tmp_res_a2, tmp_res_p1, tmp_res_p2)



In [64]:
display(coref_res)
# save to csv
coref_res.to_csv(os.path.join(RESULTS_DIR, 'coref_res_2dama.csv'), index=False)

### Plotting a line graph for fixed layer/dimensionality

In [65]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from utils.globals import *
from copy import copy

plt.rcParams['font.family'] = 'serif'
plt.rcParams['text.usetex'] = True
# plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']

#plt.rcParams['font.monospace'] = 'Ubuntu Mono'
plt.rcParams['font.size'] = 16
plt.rcParams['axes.labelsize'] = 16
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 16

sns.set_style("whitegrid")
sns.set_style({'font.family': 'Times New Roman'})

In [66]:
def plot_line_plot(res_gen, res_lm, fact_thr=None, layers=None, starred=None, save_dir=None, model_name=None):
    COLORS = sns.color_palette("colorblind", 10)

    df_gen = res_gen.copy()
    df_lm = res_lm.copy()
    df_gen['joint_slope_s'] = df_gen['joint_slope_s'] * 100
    df_gen['joint_slope_f'] = df_gen['joint_slope_f'] * 100

    reference_as = df_gen['joint_slope_s'].iloc[0]
    reference_b = df_gen['joint_slope_f'].iloc[0]
    reference_ppl = df_lm['perplexity'].iloc[0]
    if fact_thr is not None:
        df_gen = df_gen[df_gen['threshold'] == fact_thr]
        df_lm = df_lm[df_lm['threshold'] == fact_thr]
        df_lm.set_index('layers', inplace=True)
        df_gen.set_index('layers', inplace=True)
        x_label = 'layers'
        log_scale = False
    elif layers is not None:
        df_gen = df_gen[df_gen['layers'] == layers]
        df_lm = df_lm[df_lm['layers'] == layers]
        df_lm.set_index('threshold', inplace=True)
        df_gen.set_index('threshold', inplace=True)
        x_label = 'threshold'
        log_scale = True
    else:
        raise ValueError('Either dimensionality or layers must be specified')

    fig, ax = plt.subplots(1, 1, figsize=(6, 4.5))

    sns.lineplot(x=x_label, y='joint_slope_s', data=df_gen, ax=ax, label=r"$100 \cdot a_s$", color=COLORS[0])
    ax.axhline(reference_as, ls='--', color=COLORS[0])
    # put a star on the best performing model


    sns.lineplot(x=x_label, y='joint_slope_f', data=df_gen, ax=ax, label=r"$100 \cdot a_f$", color=COLORS[5])
    ax.axhline(reference_b, ls='--', color=COLORS[5])
    sns.lineplot(x=x_label, y='perplexity', data=df_lm, ax=ax, label=r"perplexity", color=COLORS[2])
    ax.axhline(reference_ppl, ls='--', color=COLORS[2])
    if starred:
        ax.scatter(starred, df_gen['joint_slope_s'].loc[starred], marker='*', s=100, color=COLORS[0])
        ax.scatter(starred, df_gen['joint_slope_f'].loc[starred], marker='*', s=100, color=COLORS[5])
        ax.scatter(starred, df_lm['perplexity'].loc[starred], marker='*', s=100, color=COLORS[2])

    df_gen.reset_index(inplace=True)
    df_lm.reset_index(inplace=True)
    ax.set_xlabel(x_label.capitalize())
    ax.set_ylabel(r"")
    if log_scale:
        ax.set_xscale('log')
        ax.set_xticks(df_gen[x_label].unique())
        ax.set_xticklabels(df_gen[x_label].unique())
        #don't show legend
        ax.legend().set_visible(False)


    if save_dir is not None:
        if fact_thr:
            save_path = os.path.join(save_dir, f"2dama_results_fact_thr_{fact_thr}_{model_name}.pdf")
        else:
            save_path = os.path.join(save_dir, f"2dama_results_layers_{layers}_{model_name}.pdf")
        plt.tight_layout()
        plt.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.show()

In [67]:
plot_line_plot(gen_res, lm_res, fact_thr=0.05, starred=12 , save_dir=OUTPUT_DIR, model_name=model_name)

In [68]:
plot_line_plot(gen_res, lm_res, layers=11, starred=0.05, save_dir=OUTPUT_DIR, model_name=model_name)